# 311 API Data

## Author: Yi Wang
## Group: Yi Wang, An Nisa Astuti

### Notebook Description: 
This notebook pulls Chicago 311 service request records from the City of Chicago’s Socrata API (v6vf-nfxy dataset) in monthly chunks from 2021–2025. It saves the raw CSVs, then concatenates and cleans them by dates and community area. We compute operational performance metrics including request volume, closure/share open, time-to-close (TTC) stats, open-age backlog stats, and % closed within 24h/72h/7d/30d. We then grouped these metrics by community_area × year × ward, and writes the results to data/311_metrics.csv.

### AI Statement
I used ChatGPT to learn a new API documentation called Socrata and used AI for understanding the psuedo code from https://dev.socrata.com/foundry/data.cityofchicago.org/v6vf-nfxy, including setting query parameters, handling pagination, and troubleshooting HTTP/API errors. 

In [22]:
import requests, pandas as pd, sqlite3
from bs4 import BeautifulSoup as bs
from sodapy import Socrata

In [23]:
url = "https://data.cityofchicago.org/resource/v6vf-nfxy.json"
response = requests.get(url)
print("Our response code is:", response.status_code) 
# check response code to make sure it works 

Our response code is: 200


### Testing with the token

In [24]:
# I am citing this from Chicago data portal 311 API documentation and socrata API code snippets (Chicago Open Data portal uses Socrata)
# On the code snippets, it has instructions and code on how to access the API data with token
# using the original template, I am building on the given template
# "https://dev.socrata.com/foundry/data.cityofchicago.org/v6vf-nfxy"
# "https://dev.socrata.com/docs/queries/page"


token = "iAZp9mxjcw3dt8SUNugTPYJbS" 
client = Socrata("data.cityofchicago.org", token, timeout = 60)

DATASET_ID = "v6vf-nfxy"

SELECT_COLS = ",".join(["sr_number",
    "sr_type",
    "sr_short_code",
    "status",
    "origin",
    "created_department",
    "owner_department",
    "created_date",
    "last_modified_date",
    "closed_date",
    "latitude",
    "longitude",
    "zip_code",
    "community_area",
    "ward",
    "duplicate",
    "legacy_record"
])

BASE_WHERE = """
created_date >= '{start}' AND created_date < '{end}'
AND duplicate = false AND legacy_record = false
"""

def fetch_311(start, end, limit=50000):
    """Fetch ALL rows for [start, end) using Socrata paging."""
    offset = 0
    out = []

    where = BASE_WHERE.format(start=start, end=end)

    while True:
        rows = client.get(
            DATASET_ID,
            select=SELECT_COLS,
            where=where,
            order="created_date ASC, sr_number ASC",
            limit=limit,
            offset=offset
        )
        if not rows:
            break

        out.append(pd.DataFrame.from_records(rows))

        if len(rows) < limit:
            break

        offset += limit

    if out:
        return pd.concat(out, ignore_index=True)
    return pd.DataFrame()


### Pulling data in chunks

In [25]:
import time

def fetch_all(where, limit=20000, max_retries=5):
    offset = 0
    frames = []

    while True:
        attempt = 0
        while True:
            try:
                rows = client.get(
                    DATASET_ID,
                    select=SELECT_COLS,
                    where=where,
                    order="created_date ASC, sr_number ASC",
                    limit=limit,
                    offset=offset
                )
                break  
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise  

                wait = 2 * attempt 
                print(f"Timeout, trying...")
                time.sleep(wait)

        if not rows:
            break

        frames.append(pd.DataFrame.from_records(rows))

        if len(rows) < limit:
            break

        offset += limit
        time.sleep(0.2) 

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

In [26]:
where_jan2023 = BASE_WHERE.format(
    start="2023-01-01T00:00:00.000",
    end="2023-02-01T00:00:00.000"
)

df_jan = fetch_all(where_jan2023, limit=50000)
print("Jan 2023 rows:", len(df_jan))
df_jan.head()

Jan 2023 rows: 123412


,sr_number,sr_type,sr_short_code,status,origin,owner_department,created_date,last_modified_date,latitude,longitude,community_area,ward,duplicate,legacy_record,closed_date,created_department,zip_code
0,SR23-00000001,Water Lead Test Kit Request,WCA2,Open,DWM,DWM - Department of Water Management,2023-01-01T00:01:17.000,2023-01-01T00:30:34.000,41.99731200094083,-87.79661100000192,10,41,False,False,NaN,NaN,NaN
1,SR23-00000002,Water Lead Test Kit Request,WCA2,Completed,DWM,DWM - Department of Water Management,2023-01-01T00:02:15.000,2023-02-27T14:55:07.000,41.99731200094083,-87.79661100000192,10,41,False,False,2023-02-27T14:55:07.000,NaN,NaN
2,SR23-00000003,Water Lead Test Kit Request,WCA2,Open,DWM,DWM - Department of Water Management,2023-01-01T00:02:39.000,2023-01-01T00:30:34.000,41.99731200094083,-87.79661100000192,10,41,False,False,NaN,NaN,NaN
3,SR23-00000004,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,2023-01-01T00:11:26.000,2023-01-01T00:30:34.000,41.87183400094043,-87.6798450000019,28,28,False,False,2023-01-01T00:11:27.000,311 City Services,60612
4,SR23-00000005,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,2023-01-01T00:11:51.000,2023-01-01T00:30:34.000,41.87183400094043,-87.6798450000019,28,28,False,False,2023-01-01T00:11:52.000,311 City Services,60612


- Missing rate of zipcode is too high, we have decided to switch from using zipcode to community area. 

In [27]:
import os
from pathlib import Path

OUT_DIR = Path("~/Downloads/311_request_data/data/raw_data").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

def next_month(y, m):
    return (y + 1, 1) if m == 12 else (y, m + 1)

for y in [2021, 2022, 2023, 2024, 2025]:
    for m in range(1, 13):
        tag  = f"{y}_{m:02d}"
        path = OUT_DIR / f"311_{tag}.csv"

        if path.exists():
            print(tag, "skipped (cached)")
            continue

        y2, m2 = next_month(y, m)
        start = f"{y}-{m:02d}-01T00:00:00.000"
        end   = f"{y2}-{m2:02d}-01T00:00:00.000"

        df_m = fetch_all(BASE_WHERE.format(start=start, end=end), limit=20000)
        df_m.to_csv(path, index=False)
        print(tag, "rows:", len(df_m))

2021_01 skipped (cached)
2021_02 skipped (cached)
2021_03 skipped (cached)
2021_04 skipped (cached)
2021_05 skipped (cached)
2021_06 skipped (cached)
2021_07 skipped (cached)
2021_08 skipped (cached)
2021_09 skipped (cached)
2021_10 skipped (cached)
2021_11 skipped (cached)
2021_12 skipped (cached)
2022_01 skipped (cached)
2022_02 skipped (cached)
2022_03 skipped (cached)
2022_04 skipped (cached)
2022_05 skipped (cached)
2022_06 skipped (cached)
2022_07 skipped (cached)
2022_08 skipped (cached)
2022_09 skipped (cached)
2022_10 skipped (cached)
2022_11 skipped (cached)
2022_12 skipped (cached)
2023_01 skipped (cached)
2023_02 skipped (cached)
2023_03 skipped (cached)
2023_04 skipped (cached)
2023_05 skipped (cached)
2023_06 skipped (cached)
2023_07 skipped (cached)
2023_08 skipped (cached)
2023_09 skipped (cached)
2023_10 skipped (cached)
2023_11 skipped (cached)
2023_12 skipped (cached)
2024_01 skipped (cached)
2024_02 skipped (cached)
2024_03 skipped (cached)
2024_04 skipped (cached)


### Data Cleaning

In [32]:
import glob
from pathlib import Path

pattern = str(Path("~/Downloads/311_request_data/data/raw_data/311_*.csv").expanduser())
files = sorted(glob.glob(pattern))

print(f"Found {len(files)} files") 

dfs = [pd.read_csv(f) for f in files]
df_311 = pd.concat(dfs, ignore_index=True)
df_311

Found 60 files


,sr_number,sr_type,sr_short_code,status,origin,created_department,owner_department,created_date,last_modified_date,closed_date,latitude,longitude,zip_code,community_area,ward,duplicate,legacy_record
0,SR21-00000001,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:00:09.000,2021-01-01T00:31:03.000,2021-01-01T00:00:09.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
1,SR21-00000002,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:00:56.000,2021-01-01T00:31:03.000,2021-01-01T00:00:56.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
2,SR21-00000004,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:01:19.000,2021-01-01T00:31:22.000,2021-01-01T00:01:19.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
3,SR21-00000005,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:02:09.000,2021-01-01T00:31:30.000,2021-01-01T00:02:09.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
4,SR21-00000006,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2021-01-01T00:02:17.000,2021-01-01T00:31:03.000,2021-01-01T00:02:17.000,41.871831,-87.679846,60612.0,28.0,28.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8867640,SR25-02388503,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2025-12-31T23:53:19.000,2026-01-01T00:31:12.000,2025-12-31T23:53:19.000,41.871834,-87.679845,60612.0,28.0,28.0,False,False
8867641,SR25-02388506,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2025-12-31T23:56:01.000,2026-01-01T00:31:27.000,2025-12-31T23:56:01.000,41.871834,-87.679845,60612.0,28.0,28.0,False,False
8867642,SR25-02388507,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2025-12-31T23:58:22.000,2026-01-01T00:31:27.000,2025-12-31T23:58:23.000,41.871834,-87.679845,60612.0,28.0,28.0,False,False
8867643,SR26-00000001,311 INFORMATION ONLY CALL,311IOC,Completed,Phone Call,311 City Services,311 City Services,2025-12-31T23:58:43.000,2026-01-01T00:31:12.000,2025-12-31T23:58:44.000,41.871834,-87.679845,60612.0,28.0,28.0,False,False


In [36]:
def add_basic_metrics_cols(df):
    df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    df["closed_date"]  = pd.to_datetime(df["closed_date"], errors="coerce")

    df["year"] = df["created_date"].dt.year

    df["is_closed"] = df["closed_date"].notna()

    df["ttc_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
    df.loc[df["ttc_hours"] < 0, "ttc_hours"] = pd.NA  

    return df


In [34]:
df_311.to_csv("~/Downloads/311_request_data/data/311_2021_to_2025_clean.csv", index=False)

In [38]:
df = pd.read_csv("~/Downloads/311_request_data/data/311_2021_to_2025_clean.csv")

df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce")
df = df.dropna(subset=["community_area"]).copy()
df["community_area"] = df["community_area"].astype(int)

df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
df["closed_date"]  = pd.to_datetime(df["closed_date"], errors="coerce")
df = df.dropna(subset=["created_date"]).copy()
df["year"] = df["created_date"].dt.year

df["is_closed"] = df["closed_date"].notna()

# TTC for closed only
df["ttc_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
df.loc[~df["is_closed"], "ttc_hours"] = pd.NA
df.loc[df["ttc_hours"] < 0, "ttc_hours"] = pd.NA

ASOF = df["created_date"].max()
df["age_open_hours"] = (ASOF - df["created_date"]).dt.total_seconds() / 3600
df.loc[df["is_closed"], "age_open_hours"] = pd.NA
df.loc[df["age_open_hours"] < 0, "age_open_hours"] = pd.NA

# Precompute closure-within indicators (open requests count as False)
df["closed_within_24h"] = df["is_closed"] & (df["ttc_hours"] <= 24)
df["closed_within_72h"] = df["is_closed"] & (df["ttc_hours"] <= 72)
df["closed_within_7d"]  = df["is_closed"] & (df["ttc_hours"] <= 24*7)
df["closed_within_30d"] = df["is_closed"] & (df["ttc_hours"] <= 24*30)

final_metrics = (
    df.groupby(["community_area", "year", "ward"], dropna=False)
      .agg(
          n_requests=("sr_number", "size"),
          n_closed=("is_closed", "sum"),
          share_closed=("is_closed", "mean"),

          n_open=("is_closed", lambda s: (~s).sum()),
          share_open=("is_closed", lambda s: (~s).mean()),

          median_ttc_hours=("ttc_hours", "median"),
          p75_ttc_hours=("ttc_hours", lambda s: s.quantile(0.75)),
          mean_ttc_hours=("ttc_hours", "mean"),

          median_age_open_hours=("age_open_hours", "median"),
          p75_age_open_hours=("age_open_hours", lambda s: s.quantile(0.75)),

          pct_closed_24h=("closed_within_24h", "mean"),
          pct_closed_72h=("closed_within_72h", "mean"),
          pct_closed_7d=("closed_within_7d", "mean"),
          pct_closed_30d=("closed_within_30d", "mean"),
      )
      .reset_index()
)

final_metrics["small_n_requests"] = final_metrics["n_requests"] < 30
final_metrics["small_n_closed"]   = final_metrics["n_closed"] < 30

# Round readable columns
to_round = [
    "share_closed", "share_open",
    "median_ttc_hours", "p75_ttc_hours", "mean_ttc_hours",
    "median_age_open_hours", "p75_age_open_hours",
    "pct_closed_24h", "pct_closed_72h", "pct_closed_7d", "pct_closed_30d",
]
for c in to_round:
    if c in final_metrics.columns:
        final_metrics[c] = final_metrics[c].round(4)

final_metrics.to_csv("~/Downloads/311_request_data/data/311_metrics.csv", index=False)
print("Saved:", final_metrics.shape)
final_metrics.head()


Saved: (1394, 19)


,community_area,year,ward,n_requests,n_closed,share_closed,n_open,share_open,median_ttc_hours,p75_ttc_hours,mean_ttc_hours,median_age_open_hours,p75_age_open_hours,pct_closed_24h,pct_closed_72h,pct_closed_7d,pct_closed_30d,small_n_requests,small_n_closed
0,1,2021,40.0,489,487,0.9959,2,0.0041,65.8528,403.3588,987.2005,40425.7935,41301.9181,0.3436,0.5215,0.6442,0.8303,False,False
1,1,2021,49.0,8226,8117,0.9867,109,0.0133,62.9389,485.9081,1065.8087,38986.7958,41413.5375,0.3820,0.5179,0.6071,0.7755,False,False
2,1,2021,50.0,24,24,1.0000,0,0.0000,0.8431,45.3127,1196.4521,NaN,NaN,0.7083,0.7917,0.8333,0.8333,True,True
3,1,2022,40.0,616,616,1.0000,0,0.0000,52.2540,345.5275,562.5324,NaN,NaN,0.3393,0.5292,0.6575,0.8360,False,False
4,1,2022,49.0,8851,8707,0.9837,144,0.0163,48.7883,402.3939,1021.6383,29754.6719,31509.0319,0.4071,0.5370,0.6228,0.7926,False,False


In [39]:
print(len(df)) # check length
print(df["created_date"].min(), "to", df["created_date"].max()) # check time period 
print("missing community_area:", df["community_area"].isna().mean()) # check missing rate
print("missing closed_date:", df["closed_date"].isna().mean()) # check missing rate for closed_date for request

final_metrics.sort_values("n_requests", ascending=False).head(10)


8858595
2021-01-01 00:00:09 to 2025-12-31 23:59:32
missing community_area: 0.0
missing closed_date: 0.013374694294072592


,community_area,year,ward,n_requests,n_closed,share_closed,n_open,share_open,median_ttc_hours,p75_ttc_hours,mean_ttc_hours,median_age_open_hours,p75_age_open_hours,pct_closed_24h,pct_closed_72h,pct_closed_7d,pct_closed_30d,small_n_requests,small_n_closed
595,28,2025,28.0,667862,667443,0.9994,419,0.0006,0.0,0.0003,3.9672,3651.4978,5670.5903,0.9943,0.9954,0.9963,0.9977,False,False
588,28,2024,28.0,661937,661586,0.9995,351,0.0005,0.0,0.0003,9.5073,13286.5506,14674.1833,0.9934,0.9948,0.9958,0.9974,False,False
569,28,2021,28.0,637367,637320,0.9999,47,0.0001,0.0,0.0003,8.7868,38957.4897,41154.3642,0.9950,0.9957,0.9965,0.9980,False,False
582,28,2023,28.0,635590,635466,0.9998,124,0.0002,0.0,0.0003,8.0490,20494.3526,22071.0794,0.9944,0.9956,0.9965,0.9983,False,False
575,28,2022,28.0,613181,613132,0.9999,49,0.0001,0.0,0.0003,7.4812,31044.9025,32770.0922,0.9947,0.9957,0.9964,0.9982,False,False
1374,76,2025,41.0,366926,366805,0.9997,121,0.0003,0.0,0.0003,2.5680,1864.8181,3218.7711,0.9964,0.9969,0.9973,0.9982,False,False
1371,76,2024,41.0,345383,345366,1.0000,17,0.0000,0.0,0.0003,5.4455,13257.5194,15970.2761,0.9963,0.9968,0.9971,0.9980,False,False
1366,76,2022,41.0,326975,326964,1.0000,11,0.0000,0.0,0.0003,3.9259,31286.1411,31954.5297,0.9968,0.9973,0.9975,0.9984,False,False
1364,76,2021,41.0,318274,318268,1.0000,6,0.0000,0.0,0.0003,4.2062,39363.5753,40996.2581,0.9971,0.9975,0.9977,0.9985,False,False
1368,76,2023,41.0,309663,309645,0.9999,18,0.0001,0.0,0.0003,3.9592,20486.6907,22084.1796,0.9962,0.9966,0.9969,0.9981,False,False


### Reading the new compiled and cleaned file

In [40]:
df2 = pd.read_csv("~/Downloads/311_request_data/data/311_metrics.csv")  

In [41]:
df2.columns

Index(['community_area', 'year', 'ward', 'n_requests', 'n_closed',
       'share_closed', 'n_open', 'share_open', 'median_ttc_hours',
       'p75_ttc_hours', 'mean_ttc_hours', 'median_age_open_hours',
       'p75_age_open_hours', 'pct_closed_24h', 'pct_closed_72h',
       'pct_closed_7d', 'pct_closed_30d', 'small_n_requests',
       'small_n_closed'],
      dtype='str')